# ASOS Multimodal Search Engine

### Part 1: Data Audit and EDA

### Part 2: Preprocessing Pipeline            

Fixes all critical bugs + extracts structured features for CLIP search

In [ ]:
!pip install -qqq pandas numpy matplotlib seaborn nltk transformers spacy chromadb kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import nltk
import transformers
import spacy
import chromadb
import ast
import re
from collections import Counter
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [ ]:
!kaggle datasets download -d trainingdatapro/asos-e-commerce-dataset-30845-products --unzip

Dataset URL: https://www.kaggle.com/datasets/trainingdatapro/asos-e-commerce-dataset-30845-products
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 7.55M/7.55M [00:00<00:00, 56.6MB/s]



In [ ]:
df_raw = pd.read_csv('/content/products_asos.csv')
df_raw.head(3)

,url,name,size,category,price,color,sku,description,images
0,https://www.asos.com/stradivarius/stradivarius...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...
1,https://www.asos.com/stradivarius/stradivarius...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...
2,https://www.asos.com/asos-design/asos-design-l...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...


In [ ]:
df_raw.shape

(30845, 9)

In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30845 entries, 0 to 30844
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   url          30827 non-null  object 
 1   name         30827 non-null  object 
 2   size         30827 non-null  object 
 3   category     30827 non-null  object 
 4   price        30827 non-null  object 
 5   color        30827 non-null  object 
 6   sku          30827 non-null  float64
 7   description  30827 non-null  object 
 8   images       30827 non-null  object 
dtypes: float64(1), object(8)
memory usage: 2.1+ MB


In [ ]:
df_raw.isnull().sum()

,0
url,18
name,18
size,18
category,18
price,18
color,18
sku,18
description,18
images,18


In [ ]:
df_raw[df_raw["url"].isnull()]

,url,name,size,category,price,color,sku,description,images
13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
339,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df = df_raw.dropna()
df.isnull().sum()

,0
url,0
name,0
size,0
category,0
price,0
color,0
sku,0
description,0
images,0


In [ ]:
df.describe()

,sku
count,3.082700e+04
mean,1.154976e+08
std,2.259336e+07
min,4.010200e+05
25%,1.173468e+08
50%,1.199234e+08
75%,1.228726e+08
max,1.292616e+08


In [ ]:
df.describe(include='object')

,url,name,size,category,price,color,description,images
count,30827,30827,30827,30827,30827,30827,30827,30827
unique,30468,29493,5073,29492,880,3636,29971,29972
top,https://www.asos.com/in-the-style/in-the-style...,ASOS 4505 icon performance t-shirt,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14,UK 16,UK 18",ASOS 4505 icon performance t-shirt,30.00,BLACK,[{'Product Details': 'Coats & Jackets by Collu...,['https://images.asos-media.com/products/collu...
freq,2,7,2912,7,706,3289,5,5


In [ ]:
for col in df.columns:
    print(f"{col}: {df.loc[0, col]}")

url: https://www.asos.com/stradivarius/stradivarius-faux-leather-biker-jacket-in-black/prd/203490700?clr=black&colourWayId=203490705
name: New Look trench coat in camel
size: UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stock,UK 16,UK 18
category: New Look trench coat in camel
price: 49.99
color: Neutral
sku: 126704571.0
description: [{'Product Details': 'Coats & Jackets by New LookLow-key layeringNotch collarButton placketTie waistRegular fitProduct Code: 126704571'}, {'Brand': 'Since setting up shop in the 60s, New Look has become a high-street classic known for creating universally loved, wardrobe-ready collections. Shop the New Look at ASOS edit, featuring everything from chic LBDs and printed dresses to all-important accessories and figure-flattering jeans (if you re anything like us, you re always on the hunt for those). While you re there, check out the label s cute-yet-classy tops and blouses for your next  jeans and a nice top  day.'}, {'Size & Fit': "Model wears: UK 8/ EU 36/ US

In [ ]:
price_examples = df['price'].dropna().unique()[:20]
print(f"Sample prices: {list(price_examples)}")
has_currency = df['price'].dropna().str.contains(r'[£$€]', regex=True).sum()
has_range = df['price'].dropna().str.contains(r'From|Now|-', regex=True).sum()
print(f"Prices with currency symbols: {has_currency:,}")
print(f"Prices with range/sale markers: {has_range:,}\n")

Sample prices: ['49.99', '59.99', '45.00', '84.95', '75.00', '56.00', '40.00', '38.00', '150.00', 'From  42.50', 'Now  72.00', 'Now  17.50', '30.00', 'Now  21.50', '79.99', '35.99', '69.99', 'Now  84.00', 'Now  37.20', '39.99']
Prices with currency symbols: 0
Prices with range/sale markers: 12,449



In [ ]:
colors = df_raw['color'].dropna().str.lower()
print(f"Total unique colors: {colors.nunique():,}")
print(f"Top 15 colors:\n{colors.value_counts().head(15).to_string()}")
multi_word = colors[colors.str.split().str.len() > 1]
print(f"\nMulti-word colors: {len(multi_word):,} ({len(multi_word)/len(colors)*100:.1f}%)")
print(f"Examples: {list(multi_word.unique()[:10])}\n")

Total unique colors: 2,929
Top 15 colors:
color
black     6558
white     1723
multi     1436
pink      1422
green     1035
blue       975
brown      788
cream      679
red        619
grey       554
khaki      549
beige      496
navy       403
orange     398
lilac      375

Multi-word colors: 7,460 (24.2%)
Examples: ['stone & green', 'dirty dark wash', 'washed blue', 'washed brown', 'dark teal', 'bright green', 'light green', 'off white', 'blue - bright', 'denim dark']



In [ ]:
sample_desc = df_raw['description'].dropna().iloc[0]
print(f"Type: {type(sample_desc)}")
print(f"First 200 chars: {sample_desc[:200]}")

try:
    parsed = ast.literal_eval(sample_desc)
    print(f"\nParsed type: {type(parsed)}")
    print(f"Number of dicts: {len(parsed)}")
    for d in parsed:
        for k, v in d.items():
            print(f"  Key: '{k}' | Value preview: '{str(v)[:80]}...'")
except Exception as e:
    print(f"Parse error: {e}")

Type: <class 'str'>
First 200 chars: [{'Product Details': 'Coats & Jackets by New LookLow-key layeringNotch collarButton placketTie waistRegular fitProduct Code: 126704571'}, {'Brand': 'Since setting up shop in the 60s, New Look has beco

Parsed type: <class 'list'>
Number of dicts: 5
  Key: 'Product Details' | Value preview: 'Coats & Jackets by New LookLow-key layeringNotch collarButton placketTie waistRe...'
  Key: 'Brand' | Value preview: 'Since setting up shop in the 60s, New Look has become a high-street classic know...'
  Key: 'Size & Fit' | Value preview: 'Model wears: UK 8/ EU 36/ US 4Model's height:  170 cm/5'7 ...'
  Key: 'Look After Me' | Value preview: 'Machine wash according to instructions on care label...'
  Key: 'About Me' | Value preview: 'Stretch, plain-woven fabricMain: 55% Polyester, 45% Elastomultiester....'


In [ ]:
sample_sizes = df_raw['size'].dropna().iloc[:5]
for i, s in enumerate(sample_sizes):
    has_oos = 'Out of stock' in str(s) or 'Out of Stock' in str(s)
    print(f"  Row {i}: OOS embedded={has_oos} | '{str(s)[:80]}...'")

oos_in_size = df_raw['size'].dropna().str.contains('Out of stock|Out of Stock', case=False, regex=True).sum()
print(f"\nRows with 'Out of stock' in size field: {oos_in_size:,}")


  Row 0: OOS embedded=True | 'UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stock,UK 16,UK 18...'
  Row 1: OOS embedded=True | 'UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stock,UK 16,UK 18...'
  Row 2: OOS embedded=True | 'UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stock,UK 16,UK 18...'
  Row 3: OOS embedded=True | 'UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stock,UK 16,UK 18...'
  Row 4: OOS embedded=False | 'XS - UK 6,S - UK 8,M - UK 10,L - UK 12,XL - UK 14...'

Rows with 'Out of stock' in size field: 17,120


In [ ]:
sample_img = df_raw['images'].dropna().iloc[0]
try:
    img_list = ast.literal_eval(sample_img)
    print(f"Parsed type: {type(img_list)}")
    print(f"Number of images per product (sample): {len(img_list)}")
    print(f"First URL: {img_list[0][:80]}...")
except Exception as e:
    print(f"Parse error: {e}")

def count_images(img_str):
    try:
        return len(ast.literal_eval(str(img_str)))
    except:
        return 0

img_counts = df_raw['images'].dropna().apply(count_images)
print(f"\nImages per product — min: {img_counts.min()}, median: {img_counts.median():.0f}, "
      f"max: {img_counts.max()}, mean: {img_counts.mean():.1f}")

Parsed type: <class 'list'>
Number of images per product (sample): 5
First URL: https://images.asos-media.com/products/new-look-trench-coat-in-camel/204351106-4...

Images per product — min: 1, median: 5, max: 6, mean: 4.9


In [ ]:
n_dup_exact = df.duplicated().sum()
n_dup_sku = df.duplicated(subset=['sku'], keep=False).sum()
n_dup_name = df.duplicated(subset=['name'], keep=False).sum()
n_dup_url = df.duplicated(subset=['url'], keep=False).sum()
print(f"Exact row duplicates:  {n_dup_exact:,}")
print(f"Duplicate SKUs:        {n_dup_sku:,} rows share a SKU with another row")
print(f"Duplicate names:       {n_dup_name:,}")
print(f"Duplicate URLs:        {n_dup_url:,}")

Exact row duplicates:  326
Duplicate SKUs:        1,379 rows share a SKU with another row
Duplicate names:       2,255
Duplicate URLs:        718


In [ ]:
# Check the URL ↔ name mismatch
sample = df.head(5)[['url', 'name', 'sku']].copy()
sample['url_product'] = sample['url'].str.extract(r'/([^/]+)/prd', expand=False)
for _, row in sample.iterrows():
    url_slug = str(row['url_product']).replace('-', ' ') if pd.notna(row['url_product']) else 'N/A'
    name = str(row['name']).lower()
    match = 'MATCH' if any(w in url_slug for w in name.split()[:3]) else '⚠️ MISMATCH'
    print(f"  SKU {row['sku']:.0f} | {match}")
    print(f"    URL :     {url_slug[:60]}")
    print(f"    Name:     {name[:60]}\n")

  SKU 126704571 | ⚠️ MISMATCH
    URL :     stradivarius faux leather biker jacket in black
    Name:     new look trench coat in camel

  SKU 126704571 | MATCH
    URL :     stradivarius trench coat in light stone
    Name:     new look trench coat in camel

  SKU 126704571 | MATCH
    URL :     asos design longline trench coat in stone
    Name:     new look trench coat in camel

  SKU 126704571 | MATCH
    URL :     new look trench coat in camel
    Name:     new look trench coat in camel

  SKU 123650194 | MATCH
    URL :     stradivarius trench coat in khaki
    Name:     stradivarius double breasted wool coat in grey



In [ ]:
df_temp = df
name_eq_cat = (df_temp['name'] == df_temp['category']).sum()
print(f"Rows where name == category: {name_eq_cat:,} / {len(df_temp):,} "
      f"({name_eq_cat/len(df_temp)*100:.1f}%)")
print("The Category column is just a copy of name.")


Rows where name == category: 30,819 / 30,827 (100.0%)
The Category column is just a copy of name.



# **Part 2: Preprocessing**

In [ ]:
df = df_raw.dropna(how='any').reset_index(drop=True)

FIX CRITICAL BUG 1 & 3: URL and NAME MISMATCH + SMART DEDUPLICATION

============================================================================
Strategy:
 1. Extract the product slug from the URL
 2. Compare URL slug tokens against the product name
 3. Flag mismatches — these are corrupted rows
 4. For duplicate SKUs, keep the row where URL matches name best
 5. If NO row matches for a SKU, keep the one with the most metadata consistency

In [ ]:
def extract_url_slug(url: str) -> str:
    """Extract the human-readable product slug from an ASOS URL."""
    if pd.isna(url):
        return ''
    match = re.search(r'/([^/]+)/prd', str(url))
    if match:
        return match.group(1).replace('-', ' ').lower()
    # Fallback: grab the last path segment before query params
    path = str(url).split('?')[0].rstrip('/')
    return path.split('/')[-1].replace('-', ' ').lower()

def compute_name_url_similarity(name: str, url_slug: str) -> float:
    """Token-overlap similarity between product name and URL slug."""
    if not name or not url_slug:
        return 0.0
    name_tokens = set(str(name).lower().split())
    slug_tokens = set(str(url_slug).split())
    # Remove very common stopwords
    stopwords = {'in', 'the', 'a', 'an', 'and', 'or', 'for', 'with', 'to', 'of', 'by'}
    name_tokens -= stopwords
    slug_tokens -= stopwords
    if not name_tokens or not slug_tokens:
        return 0.0
    intersection = name_tokens & slug_tokens
    # Jaccard-like but weighted toward name coverage
    return len(intersection) / max(len(name_tokens), 1)

# Extract URL slugs and compute similarity scores
df['_url_slug'] = df['url'].apply(extract_url_slug)
df['_name_url_sim'] = df.apply(
    lambda r: compute_name_url_similarity(r['name'], r['_url_slug']), axis=1
)

# Report mismatch severity
sim_bins = pd.cut(df['_name_url_sim'], bins=[0, 0.1, 0.3, 0.5, 1.0],
                  labels=['Zero', 'Low', 'Medium', 'High'], include_lowest=True)
print(f"\n── URL ↔ NAME SIMILARITY DISTRIBUTION ──")
print(sim_bins.value_counts().to_string())

# Smart deduplication: for each SKU, keep the row with highest name↔URL similarity
# This ensures we keep the most internally-consistent record
df = df.sort_values('_name_url_sim', ascending=False)
df = df.drop_duplicates(subset=['sku'], keep='first').reset_index(drop=True)
print(f"\nAfter smart SKU dedup: {df.shape[0]:,} rows")

# Flag remaining mismatches (sim < 0.3) — these have corrupted URLs
df['_url_mismatch'] = df['_name_url_sim'] < 0.3
n_mismatch = df['_url_mismatch'].sum()
print(f"Rows with URL↔name mismatch (sim < 0.3): {n_mismatch:,} "
      f"({n_mismatch/len(df)*100:.1f}%)")
print("→ These rows will use name-based image search instead of URL-linked images\n")

# Clean up temp columns
df = df.drop(columns=['_url_slug', '_name_url_sim'])


── URL ↔ NAME SIMILARITY DISTRIBUTION ──
_name_url_sim
High      30308
Zero        239
Low         221
Medium       59

After smart SKU dedup: 29,971 rows
Rows with URL↔name mismatch (sim < 0.3): 5 (0.0%)
→ These rows will use name-based image search instead of URL-linked images



2.3  PARSE DESCRIPTION — EXTRACT STRUCTURED FIELDS

============================================================================
The description column is a stringified list of dicts with keys like:
 'Product Details', 'Brand', 'Size & Fit', 'Look After Me', 'About Me'

In [ ]:
def safe_parse_description(desc_str: str) -> dict:
    """Parse the stringified list-of-dicts into a clean dictionary."""
    if pd.isna(desc_str):
        return {}
    try:
        parsed = ast.literal_eval(str(desc_str))
        if isinstance(parsed, list):
            merged = {}
            for item in parsed:
                if isinstance(item, dict):
                    merged.update(item)
            return merged
        elif isinstance(parsed, dict):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return {}

def extract_product_details(desc_dict: dict) -> str:
    """Extract the 'Product Details' text, cleaned."""
    raw = desc_dict.get('Product Details', '')
    # Remove product code
    raw = re.sub(r'Product Code:\s*\d+', '', raw)
    # Add spaces before capital letters that are jammed together
    # e.g., "New LookLow-key" → "New Look Low-key"
    raw = re.sub(r'([a-z])([A-Z])', r'\1 \2', raw)
    return raw.strip()

def extract_fabric_info(desc_dict: dict) -> str:
    """Extract fabric/material from 'About Me' section."""
    raw = desc_dict.get('About Me', '')
    # Look for percentage-based fabric composition
    fabrics = re.findall(r'(\d+%?\s*[A-Za-z]+(?:\s+[A-Za-z]+)?)', raw)
    if fabrics:
        return ', '.join(fabrics)
    return raw.strip()

def extract_fabric_materials(desc_dict: dict) -> list:
    """Extract individual material names for tagging."""
    raw = desc_dict.get('About Me', '')
    materials = re.findall(r'\d+%\s*([A-Za-z]+(?:\s+[A-Za-z]+)?)', raw)
    return [m.strip().lower() for m in materials if len(m.strip()) > 1]

def extract_fit_info(desc_dict: dict) -> str:
    """Extract fit description from 'Size & Fit'."""
    raw = desc_dict.get('Size & Fit', '')
    return raw.strip()

def extract_care_info(desc_dict: dict) -> str:
    """Extract care instructions from 'Look After Me'."""
    raw = desc_dict.get('Look After Me', '')
    return raw.strip()

def extract_brand_story(desc_dict: dict) -> str:
    """Extract brand description from 'Brand' section."""
    raw = desc_dict.get('Brand', '')
    return raw.strip()

# Parse all descriptions
print("── PARSING DESCRIPTIONS ──")
df['_desc_parsed'] = df['description'].apply(safe_parse_description)

df['product_details'] = df['_desc_parsed'].apply(extract_product_details)
df['fabric_raw']      = df['_desc_parsed'].apply(extract_fabric_info)
df['materials']       = df['_desc_parsed'].apply(extract_fabric_materials)
df['fit_info']        = df['_desc_parsed'].apply(extract_fit_info)
df['care_info']       = df['_desc_parsed'].apply(extract_care_info)
df['brand_story']     = df['_desc_parsed'].apply(extract_brand_story)

# Report extraction rates
for col in ['product_details', 'fabric_raw', 'fit_info', 'care_info', 'brand_story']:
    non_empty = (df[col].str.len() > 0).sum()
    print(f"  {col}: {non_empty:,} / {len(df):,} extracted ({non_empty/len(df)*100:.1f}%)")

df = df.drop(columns=['_desc_parsed'])
print()

── PARSING DESCRIPTIONS ──
  product_details: 29,969 / 29,971 extracted (100.0%)
  fabric_raw: 29,969 / 29,971 extracted (100.0%)
  fit_info: 29,025 / 29,971 extracted (96.8%)
  care_info: 29,774 / 29,971 extracted (99.3%)
  brand_story: 29,219 / 29,971 extracted (97.5%)



2.4  PARSE IMAGES — EXTRACT INDIVIDUAL URLS

In [ ]:
def safe_parse_images(img_str: str) -> list:
    """Parse the stringified list of image URLs."""
    if pd.isna(img_str):
        return []
    try:
        parsed = ast.literal_eval(str(img_str))
        if isinstance(parsed, list):
            return [url.strip() for url in parsed if isinstance(url, str) and url.startswith('http')]
        return []
    except (ValueError, SyntaxError):
        # Fallback: regex extract URLs
        urls = re.findall(r"'(https?://[^']+)'", str(img_str))
        return urls

df['image_urls'] = df['images'].apply(safe_parse_images)
df['n_images']   = df['image_urls'].apply(len)
df['primary_image_url'] = df['image_urls'].apply(lambda x: x[0] if x else None)

print("── IMAGE PARSING ──")
print(f"Products with ≥1 image: {(df['n_images'] > 0).sum():,}")
print(f"Products with 0 images: {(df['n_images'] == 0).sum():,}")
print(f"Mean images per product: {df['n_images'].mean():.1f}")
print(f"Median: {df['n_images'].median():.0f}, Max: {df['n_images'].max()}\n")

── IMAGE PARSING ──
Products with ≥1 image: 29,971
Products with 0 images: 0
Mean images per product: 4.9
Median: 5, Max: 6



2.5  FIX CRITICAL BUG #2: EXTRACT REAL CATEGORIES FROM PRODUCT NAMES

============================================================================

Since category == name, we must build our own taxonomy.
Strategy: keyword matching against a curated fashion taxonomy.

In [ ]:
df[df["name"].str.contains("Vero Moda Curve skinny jean in dark blue")]["primary_image_url"].values

array(['https://images.asos-media.com/products/vero-moda-curve-skinny-jean-in-dark-blue/204104017-4?$n_1920w$&wid=1926&fit=constrain'],
      dtype=object)

In [ ]:
FASHION_TAXONOMY = {
    # Tops
    't-shirt':     'Tops',
    'tee':         'Tops',
    'shirt':       'Tops',
    'blouse':      'Tops',
    'crop top':    'Tops',
    'tank top':    'Tops',
    'vest':        'Tops',
    'cami':        'Tops',
    'bodysuit':    'Tops',
    'polo':        'Tops',
    'henley':      'Tops',
    'tunic':       'Tops',
    'top':         'Tops',
    'bralet':      'Tops',
    'bralette':    'Tops',
    'corset':      'Tops',
    'bustier':     'Tops',

    # Knitwear & Sweaters
    'jumper':      'Knitwear',
    'sweater':     'Knitwear',
    'cardigan':    'Knitwear',
    'pullover':    'Knitwear',
    'knit':        'Knitwear',
    'turtleneck':  'Knitwear',
    'roll neck':   'Knitwear',

    # Hoodies & Sweatshirts
    'hoodie':      'Hoodies & Sweatshirts',
    'sweatshirt':  'Hoodies & Sweatshirts',
    'zip-up':      'Hoodies & Sweatshirts',

    # Dresses
    'dress':       'Dresses',
    'gown':        'Dresses',
    'maxi dress':  'Dresses',
    'midi dress':  'Dresses',
    'mini dress':  'Dresses',
    'slip dress':  'Dresses',
    'bodycon':     'Dresses',
    'smock dress': 'Dresses',
    'sundress':   'Dresses',

    # Coats & Jackets
    'coat':        'Coats & Jackets',
    'jacket':      'Coats & Jackets',
    'blazer':      'Coats & Jackets',
    'parka':       'Coats & Jackets',
    'trench':      'Coats & Jackets',
    'bomber':      'Coats & Jackets',
    'puffer':      'Coats & Jackets',
    'gilet':       'Coats & Jackets',
    'anorak':      'Coats & Jackets',
    'mac ':        'Coats & Jackets',
    'windbreaker': 'Coats & Jackets',
    'overcoat':    'Coats & Jackets',
    'cagoule':     'Coats & Jackets',
    'fleece':      'Coats & Jackets',
    'shaket':     'Coats & Jackets',

    # Jeans
    'jeans':       'Jeans',
    'jean':       'Jeans',
    'denim':       'Jeans',
    'jegging':     'Jeans',

    # Trousers & Pants
    'trouser':     'Trousers',
    'chino':       'Trousers',
    'pant':        'Trousers',
    'jogger':      'Trousers',
    'legging':     'Trousers',
    'cargo':       'Trousers',
    'culottes':    'Trousers',
    'wide leg':    'Trousers',
    'flare':       'Trousers',

    # Shorts
    'shorts':      'Shorts',
    'short':       'Shorts',

    # Skirts
    'skirt':       'Skirts',
    'mini skirt':  'Skirts',
    'midi skirt':  'Skirts',
    'maxi skirt':  'Skirts',
    'pleated skirt': 'Skirts',
    'skort':      'Skirts',

    # Jumpsuits & Playsuits
    'jumpsuit':    'Jumpsuits & Playsuits',
    'playsuit':    'Jumpsuits & Playsuits',
    'romper':      'Jumpsuits & Playsuits',
    'dungaree':    'Jumpsuits & Playsuits',
    'overalls':    'Jumpsuits & Playsuits',
    'boilersuit': 'Jumpsuits & Playsuits',

    # Suits & Tailoring
    'suit':        'Suits & Tailoring',
    'waistcoat':   'Suits & Tailoring',

    # Swimwear
    'bikini':      'Swimwear',
    'swimsuit':    'Swimwear',
    'swim':        'Swimwear',
    'trunks':      'Swimwear',
    'boardshorts': 'Swimwear',

    # Activewear
    'sports bra':  'Activewear',
    'legging':     'Activewear',
    'running':     'Activewear',
    'training':    'Activewear',
    'performance': 'Activewear',
    'gym':         'Activewear',
    'yoga':        'Activewear',

    # Underwear & Loungewear
    'boxers':      'Underwear & Loungewear',
    'briefs':      'Underwear & Loungewear',
    'bra':         'Underwear & Loungewear',
    'thong':       'Underwear & Loungewear',
    'pyjama':      'Underwear & Loungewear',
    'pajama':      'Underwear & Loungewear',
    'robe':        'Underwear & Loungewear',
    'lounge':      'Underwear & Loungewear',
    'nightwear':   'Underwear & Loungewear',
    'socks':       'Underwear & Loungewear',
    'sock':        'Underwear & Loungewear',
    'slippers':    'Underwear & Loungewear',

    # Shoes & Footwear
    'trainer':     'Shoes',
    'sneaker':     'Shoes',
    'boot':        'Shoes',
    'shoe':        'Shoes',
    'sandal':      'Shoes',
    'heel':        'Shoes',
    'loafer':      'Shoes',
    'mule':        'Shoes',
    'espadrille':  'Shoes',
    'brogue':      'Shoes',
    'slipper':     'Shoes',
    'slide':       'Shoes',
    'flip flop':   'Shoes',
    'platform':    'Shoes',
    'court shoe':  'Shoes',
    'derby':       'Shoes',
    'oxford':      'Shoes',
    'chelsea boot':'Shoes',
    'ankle boot':  'Shoes',

    # Accessories
    'bag':         'Accessories',
    'handbag':     'Accessories',
    'backpack':    'Accessories',
    'purse':       'Accessories',
    'wallet':      'Accessories',
    'belt':        'Accessories',
    'scarf':       'Accessories',
    'hat':         'Accessories',
    'cap':         'Accessories',
    'beanie':      'Accessories',
    'gloves':      'Accessories',
    'sunglasses':  'Accessories',
    'watch':       'Accessories',
    'tie':         'Accessories',
    'umbrella':    'Accessories',
    'headband':    'Accessories',

    # Jewellery
    'necklace':    'Jewellery',
    'bracelet':    'Jewellery',
    'ring':        'Jewellery',
    'earring':     'Jewellery',
    'chain':       'Jewellery',
    'pendant':     'Jewellery',
    'anklet':      'Jewellery',
    'cuff':        'Jewellery',
    'choker': 'Jewellery',

    # Beauty & Grooming (ASOS sells these too)
    'moisturiser': 'Beauty',
    'cleanser':    'Beauty',
    'serum':       'Beauty',
    'mask':        'Beauty',
    'lipstick':    'Beauty',
    'foundation':  'Beauty',
    'concealer':   'Beauty',
    'mascara':     'Beauty',
    'eyeliner':    'Beauty',
    'palette':     'Beauty',
    'fragrance':   'Beauty',
    'perfume':     'Beauty',
    'cologne':     'Beauty',
    'shampoo':     'Beauty',
    'conditioner': 'Beauty',
    'body wash':   'Beauty',
    'deodorant':   'Beauty',
}

# Sort by descending key length so longer/more specific matches take priority
# e.g., "slip dress" before "dress", "mini skirt" before "skirt"
SORTED_TAXONOMY = sorted(FASHION_TAXONOMY.items(), key=lambda x: -len(x[0]))

def classify_category(name: str) -> str:
    """Extract the real product category from the product name."""
    if pd.isna(name):
        return 'Other'
    name_lower = str(name).lower()
    for keyword, category in SORTED_TAXONOMY:
        # Use word boundary matching to avoid false positives
        # e.g., "ring" shouldn't match "during"
        if re.search(r'\b' + re.escape(keyword) + r's?\b', name_lower):
            return category
    return 'Other'

df['category_extracted'] = df['name'].apply(classify_category)

print("── CATEGORY EXTRACTION RESULTS ──")
cat_counts = df['category_extracted'].value_counts()
print(cat_counts.to_string())
print(f"\nProducts categorized: {(df['category_extracted'] != 'Other').sum():,} / {len(df):,} "
      f"({(df['category_extracted'] != 'Other').sum()/len(df)*100:.1f}%)")
other_examples = df[df['category_extracted'] == 'Other']['name'].head(10).tolist()
print(f"\nSample 'Other' products (need manual review):")
for ex in other_examples:
    print(f"  • {ex}")
print()

── CATEGORY EXTRACTION RESULTS ──
category_extracted
Dresses                   10024
Tops                       8481
Coats & Jackets            3883
Knitwear                   2476
Trousers                   1717
Other                       924
Underwear & Loungewear      768
Activewear                  581
Skirts                      286
Shorts                      260
Jeans                       174
Hoodies & Sweatshirts       111
Accessories                  91
Swimwear                     66
Jewellery                    55
Jumpsuits & Playsuits        38
Suits & Tailoring            29
Beauty                        4
Shoes                         3

Products categorized: 29,047 / 29,971 (96.9%)

Sample 'Other' products (need manual review):
  • New Look strappy sundress in pink
  • ASOS 4505 Curve active skort
  • ASOS DESIGN Curve herringbone shacket in oatmeal
  • ASOS DESIGN Curve long sleeve twill boilersuit with collar in burgundy
  • Native Youth Plus bold floral belted boile

In [ ]:
df[df['category_extracted'] == 'Other']['name'].sample(10).tolist()

['I Saw It First zip detail body in black',
 'ASOS DESIGN cardi with collar and button front in oatmeal',
 'ASOS DESIGN washed oversized tank with drop arm hole in washed charcoal grey',
 'COLLUSION faux wool shacket in checked print',
 'Cotton:On Maternity 3-pack everyday tank',
 'ASOS DESIGN Curve 30 denier polka dot and heart tights in black',
 'ASOS DESIGN square neck shirred mini sundress in mono spot print',
 'Ivory Rose Fuller Bust satin teddy with lace trim in white',
 'Spanx Curve Seamless Shaping boyshort in beige',
 'New Look dogtooth check shacket in pink']

2.6  EXTRACT BRAND FROM PRODUCT NAME

 ============================================================================

ASOS product names follow patterns like:
  "ASOS DESIGN oversized t-shirt" → brand = "ASOS DESIGN"

  "New Look trench coat in camel" → brand = "New Look"
  
  "Nike Air Force 1" → brand = "Nike"

In [ ]:
KNOWN_BRANDS = [
    'ASOS DESIGN', 'ASOS 4505', 'ASOS EDITION', 'ASOS LUXE', 'ASOS WHITE',
    'ASOS MADE IN', 'ASOS Curve', 'COLLUSION', 'Reclaimed Vintage',
    'New Look', 'Topshop', 'Topman', 'Stradivarius', 'Bershka', 'Pull&Bear',
    'Nike', 'adidas', 'Puma', 'Reebok', 'Under Armour',
    'The North Face', 'Dr Martens', 'Converse', 'Vans', 'New Balance',
    "Levi's", 'Tommy Hilfiger', 'Calvin Klein', 'Polo Ralph Lauren',
    'Jack & Jones', 'River Island', 'French Connection', 'Ted Baker',
    'AllSaints', 'Weekday', 'Monki', 'Mango', 'Missguided',
    'South Beach', 'Brave Soul', 'Only', 'Vero Moda', 'Noisy May',
    'Pieces', 'JDY', 'Vila', 'Y.A.S', 'Selected', 'Skechers',
    'Birkenstock', 'UGG', 'Crocs', 'Fila', 'Timberland',
    'Carhartt WIP', 'Dickies', 'Columbia', 'Berghaus',
    'In The Style', 'PrettyLittleThing', 'Boohoo',
    'Glamorous', 'Rare London', 'True Religion', 'Wrangler',
    'Lee', 'G-Star', 'G-Star RAW', 'Superdry', 'Barbour',
    'Lacoste', 'Fred Perry', 'Ben Sherman', 'Paul Smith',
    'Hugo', 'BOSS', 'Versace', 'Moschino', 'Love Moschino',
    'Michael Kors', 'Coach', 'Kate Spade', 'Marc Jacobs',
    'Ray-Ban', 'Oakley', 'Emporio Armani', 'Armani Exchange',
    'Miss Selfridge', 'Only & Sons', 'ASOS Weekend Collective',
    'Object', 'Warehouse', 'Extro & Vert',
    'Threadbare', 'Bolongaro Trevor', 'Heart & Dagger',
    'Liquor N Poker', 'Religion', 'Fashion Union',
    'Native Youth', 'Chi Chi London', 'Forever New',
    'Influence', 'Parisian', 'Lipsy', 'Closet London',
    "Wednesday's Girl", '& Other Stories', 'Mamalicious',
    'Lost Ink', 'Urban Bliss', 'Hope & Ivy',
    'Little Mistress', 'TFNC', 'Club L London'
]

KNOWN_BRANDS_SORTED = sorted(set(KNOWN_BRANDS), key=len, reverse=True)

# MUST be defined before function
STOPWORDS = {
    't-shirt', 'shirt', 'top', 'hoodie', 'sweatshirt', 'jumper', 'sweater',
    'dress', 'jeans', 'trousers', 'pants', 'shorts', 'skirt', 'coat',
    'jacket', 'blazer', 'cardigan', 'bra', 'briefs', 'bodysuit',
    'leggings', 'joggers', 'boots', 'trainers', 'sneakers', 'sandals',
    'heels', 'loafers', 'mules', 'sliders', 'bag', 'backpack', 'wallet',
    'belt', 'ring', 'necklace', 'earrings', 'bracelet', 'watch',
    'sunglasses', 'cap', 'hat', 'beanie', 'scarf', 'gloves',
    'swimsuit', 'bikini', 'sock', 'socks',
    'in', 'with', 'at', 'for', 'from', 'on', 'by', 'to',
    'oversized', 'cropped', 'slim', 'regular', 'wide', 'skinny',
    'midi', 'mini', 'maxi', 'high', 'low', 'chunky', 'classic',
    'printed', 'embroidered', 'washed', 'tailored', 'ribbed',
    'denim', 'cotton', 'linen', 'leather', 'mesh'
}

def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s)).strip()

def clean_brand_text(s):
    return re.sub(r'\s+', ' ', s).strip(" ,-")

def extract_brand(name):
    if pd.isna(name):
        return 'Unknown'

    name_str = normalize_text(name)
    lower_name = name_str.lower()

    # 1. known brands
    for brand in KNOWN_BRANDS_SORTED:
        if lower_name.startswith(brand.lower()):
            return brand

    # 2. build brand token by token
    tokens = name_str.split()
    brand_tokens = []

    for token in tokens:
        token_clean = token.strip(",.-()[]{}:;!?").lower()

        if token_clean in STOPWORDS:
            break

        if re.match(r"^[A-Za-z0-9&'./-]+$", token):
            brand_tokens.append(token)
        else:
            break

        if len(brand_tokens) >= 4:
            break

    if brand_tokens:
        candidate = clean_brand_text(" ".join(brand_tokens))
        if len(candidate) > 1:
            return candidate

    # 3. regex fallback
    match = re.match(r"^([A-Za-z0-9&'./-]+(?:\s+[A-Za-z0-9&'./-]+){0,3})", name_str)
    if match:
        return clean_brand_text(match.group(1))

    return 'Unknown'

# Apply
df['brand'] = df['name'].apply(extract_brand)

print("── BRAND EXTRACTION RESULTS ──")
print(f"Unique brands: {df['brand'].nunique()}")
print(f"Top 15 brands:\n{df['brand'].value_counts().head(15).to_string()}")
print(f"Unknown brands: {(df['brand'] == 'Unknown').sum():,}")

── BRAND EXTRACTION RESULTS ──
Unique brands: 4205
Top 15 brands:
brand
ASOS DESIGN        6894
Topshop            1792
River Island        906
Miss Selfridge      669
adidas              637
Nike                631
COLLUSION           629
New Look            608
Monki               584
Vero Moda           546
Bershka             480
ASOS EDITION        436
Stradivarius        431
Only                413
& Other Stories     352
Unknown brands: 0


In [ ]:
""" Known ASOS brands (most common)

KNOWN_BRANDS = [
    'ASOS DESIGN', 'ASOS 4505', 'ASOS EDITION', 'ASOS LUXE', 'ASOS WHITE',
    'ASOS MADE IN', 'ASOS Curve', 'COLLUSION', 'Reclaimed Vintage',
    'New Look', 'Topshop', 'Topman', 'Stradivarius', 'Bershka', 'Pull&Bear',
    'Nike', 'adidas', 'Adidas', 'Puma', 'Reebok', 'Under Armour',
    'The North Face', 'Dr Martens', 'Converse', 'Vans', 'New Balance',
    'Levi\'s', 'Tommy Hilfiger', 'Calvin Klein', 'Polo Ralph Lauren',
    'Jack & Jones', 'River Island', 'French Connection', 'Ted Baker',
    'AllSaints', 'Weekday', 'Monki', 'Mango', 'Missguided',
    'South Beach', 'Brave Soul', 'Only', 'Vero Moda', 'Noisy May',
    'Pieces', 'JDY', 'Vila', 'Y.A.S', 'Selected', 'Skechers',
    'Birkenstock', 'UGG', 'Crocs', 'Fila', 'Timberland',
    'Carhartt WIP', 'Dickies', 'Columbia', 'Berghaus',
    'In The Style', 'PrettyLittleThing', 'Boohoo', 'Missguided',
    'Glamorous', 'Rare London', 'True Religion', 'Wrangler',
    'Lee', 'G-Star', 'G-Star RAW', 'Superdry', 'Barbour',
    'Lacoste', 'Fred Perry', 'Ben Sherman', 'Paul Smith',
    'Hugo', 'BOSS', 'Versace', 'Moschino', 'Love Moschino',
    'Michael Kors', 'Coach', 'Kate Spade', 'Marc Jacobs',
    'Ray-Ban', 'Oakley', 'Emporio Armani', 'Armani Exchange',
]
"""

" Known ASOS brands (most common)\n\nKNOWN_BRANDS = [\n    'ASOS DESIGN', 'ASOS 4505', 'ASOS EDITION', 'ASOS LUXE', 'ASOS WHITE',\n    'ASOS MADE IN', 'ASOS Curve', 'COLLUSION', 'Reclaimed Vintage',\n    'New Look', 'Topshop', 'Topman', 'Stradivarius', 'Bershka', 'Pull&Bear',\n    'Nike', 'adidas', 'Adidas', 'Puma', 'Reebok', 'Under Armour',\n    'The North Face', 'Dr Martens', 'Converse', 'Vans', 'New Balance',\n    'Levi's', 'Tommy Hilfiger', 'Calvin Klein', 'Polo Ralph Lauren',\n    'Jack & Jones', 'River Island', 'French Connection', 'Ted Baker',\n    'AllSaints', 'Weekday', 'Monki', 'Mango', 'Missguided',\n    'South Beach', 'Brave Soul', 'Only', 'Vero Moda', 'Noisy May',\n    'Pieces', 'JDY', 'Vila', 'Y.A.S', 'Selected', 'Skechers',\n    'Birkenstock', 'UGG', 'Crocs', 'Fila', 'Timberland',\n    'Carhartt WIP', 'Dickies', 'Columbia', 'Berghaus',\n    'In The Style', 'PrettyLittleThing', 'Boohoo', 'Missguided',\n    'Glamorous', 'Rare London', 'True Religion', 'Wrangler',\n    'Lee

In [ ]:
"""# Sort by length descending so "ASOS DESIGN" matches before "ASOS"""
KNOWN_BRANDS_SORTED = sorted(KNOWN_BRANDS, key=len, reverse=True)

def extract_brand(name: str) -> str:
    """Extract brand name from the product name string."""
    if pd.isna(name):
        return 'Unknown'
    name_str = str(name)
    for brand in KNOWN_BRANDS_SORTED:
        if name_str.lower().startswith(brand.lower()):
            return brand
        # Also check with case-insensitive contains at start
        if re.match(re.escape(brand), name_str, re.IGNORECASE):
            return brand
    # Heuristic: first 1-3 capitalized words before a lowercase word
    match = re.match(r'^((?:[A-Z][A-Za-z&\'\.]+\s*){1,3})(?=[a-z])', name_str)
    if match:
        return match.group(1).strip()
    return 'Unknown'

df['brand'] = df['name'].apply(extract_brand)
print("── BRAND EXTRACTION RESULTS ──")
print(f"Unique brands: {df['brand'].nunique()}")
print(f"Top 15 brands:\n{df['brand'].value_counts().head(15).to_string()}")
print(f"Unknown brands: {(df['brand'] == 'Unknown').sum():,}\n")

── BRAND EXTRACTION RESULTS ──
Unique brands: 872
Top 15 brands:
brand
ASOS DESIGN       6894
Topshop           1792
River Island       906
Miss Selfridge     669
adidas             637
Nike               631
COLLUSION          629
New Look           608
Monki              584
Vero Moda          546
Unknown            518
Bershka            480
ASOS EDITION       436
Stradivarius       431
Only               413
Unknown brands: 518



In [ ]:
df[df['brand'] == 'Unknown']['name'].sample(10).tolist()

['NA-KD braided knitted sweater in brown',
 'NA-KD smock detail crop top in black',
 'ellesse swirl print flares in purple',
 "NA-KD x Mimi AR strap detail 90's midi dress in black floral",
 '4th & Reckless Tall blazer dress in chocolate',
 'NA-KD x Rianne Meijer polo kneck knitted mini dress in red',
 'NA-KD relaxed trousers in sequin co-ord',
 'NA-KD x Melissa Bentsen embroidered mesh underwired bra in green',
 'I Saw It First Plus oversized tailored blazer in black',
 'NA-KD X Josefine HJ cut out maxi dress in black']

2.7  EXTRACT GENDER / TARGET AUDIENCE

In [ ]:
def extract_gender(row) -> str:
   """Infer target gender from product name and URL."""
   name_lower = str(row['name']).lower()
   url_lower = str(row.get('url', '')).lower()
   combined = name_lower + ' ' + url_lower


   if any(kw in combined for kw in [
       'unisex', 'gender neutral', 'all gender', 'all-gender'
   ]):
       return 'Unisex'


   # Strong signals
   if any(kw in combined for kw in [
       'women', 'woman', 'maternity', 'ladies', 'female', 'girls',
       'femme', 'her ', "her's", 'herself', 'she/her',
       'curve', 'petite', 'plus size', 'plus'
   ]):
       return 'Women'
   if any(kw in combined for kw in [
       ' men ', "men's", ' mens ', '/men/', 'menswear', 'mens',
       ' man ', "man's", '/man/', 'manswear',
       'male', 'boys', 'boyfriend', 'homme',
       'him ', ' his ', 'himself', 'he/him'
   ]):
       return 'Men'
   # Weaker signals from product types
   if any(kw in name_lower for kw in [
   'bra ', 'bralet', 'bralette', 'bikini', 'dress', 'skirt', 'blouse', 'tunic',
       'leggings', 'heels', 'pumps', 'stiletto', 'camisole', 'cami',
       'bodycon', 'bandeau', 'halter', 'scrunchie', 'handbag', 'clutch',
       'thong', 'brief', 'tanga', 'bardot', 'ruched', 'shirred',
       'keyhole', 'bodice', 'off shoulder', 'off the shoulder',
       'crop top', 'cropped', 'cheeky', 'high waisted',
       'balloon sleeves', 'lettuce hem', 'broderie',
       'wrap top', 'kimono', 'maxi', 'basque', 'cupped',
       'corset', 'corset top', 'corset shirt', 'suspender detail',
       'one shoulder', 'ruffle', 'frill', 'frill neck', 'peplum',
       'angel sleeve', 'batwing sleeve', 'volume sleeve',
       'fitted shirt', 'wrap bodysuit', 'body in ', 'bodysuit',
       'tie front', 'tie strap', 'lace hem', 'hook and eye',
       'ruching', 'front ruching', 'sequin wrap', 'scoop basque',
   'brazilian', 'brazillian', 'strappy tie top', 'contouring', 'control short with lace'
   ]):
       return 'Women'
   if any(kw in name_lower for kw in [
       'boxers', 'jockstrap', 'trunks', 'briefs', 'chino', 'chinos',
       'waistcoat', 'tuxedo', 'cufflinks', 'big & tall', 'big and tall',
       'polo', 'crew neck', 'crewneck', 'overshirt', 'cargo',
       'bomber', 'windbreaker', 'quarter zip', '1/4 zip', '1/2 zip',
       'track jacket', 'track pants', 'track top',
       'football', 'academy', 'tiro',
       'muscle fit', 'shacket', 'chore coat', 'duck canvas',
       'dad denim jacket'
   ]):
       return 'Men'
   # Brand-based inference
   women_brands = {
       'missguided', 'miss selfridge', 'prettylittlething', 'monki',
       'asos design curve', 'bershka women', 'stradivarius', 'mango',
       'h&m women', 'topshop', 'topshop curve', 'simply be', 'vila',
       'vero moda', 'y.a.s', 'yas', 'tammy girl', 'na-kd',
       'the frolic', 'in the style plus', 'aria cove', 'pimkie',
       'pieces', 'pieces premium', 'morgan', 'new look curve', 'yours',
       'weekday cat', 'asyou', 'spanx', 'pretty lavish',
       'only curve', 'only petite', 'something new curve',
       'gilly hicks', 'i saw it first plus', 'free people', 'lost ink', 'pink soda', 'bailey rose'
   }
   men_brands = {
       'jack & jones', 'topman', 'ben sherman', 'burton', 'asos design men',
       'bershka men', 'pull&bear men', 'river island men',
       'carhartt wip', 'huf', 'dickies'
   }
   brand_lower = str(row.get('brand', '')).lower()
   if brand_lower in women_brands:
       return 'Women'
   if brand_lower in men_brands:
       return 'Men'
   return 'Unisex'


df['gender'] = df.apply(extract_gender, axis=1)
print("── GENDER EXTRACTION ──")
print(df['gender'].value_counts().to_string())
print()

── GENDER EXTRACTION ──
gender
Women     21967
Unisex     6674
Men        1330



In [ ]:
df[df['gender'] == 'Unisex']['name'].sample(10).tolist()

['Urban Revivo cardigan in purple',
 'Loungeable sherpa twosie nightwear set in animal print',
 '4th & Reckless tailored trouser co ord in colour block ecru  & mint',
 'Envii Lexington blazer co-ord in straw',
 'ASOS DESIGN long sleeve t-shirt with wide sleeve in mixed mono stripe',
 'ASOS 4505 Tall long sleeve training top in rib co ord',
 'Reclaimed Vintage unisex t-shirt in washed khaki',
 'Puma long sleeve top with tie back details in sage green',
 'Nike Training One Luxe Dri-FIT twist hem t-shirt in mint green',
 'ASOS DESIGN boxy tee shirt in pink and green stripe']

2.8  NORMALIZE COLORS — MAP 2,929 → ~25 CANONICAL FAMILIES

============================================================================

In [ ]:
CANONICAL_COLORS = {
    # Black family
    'black':     ['black', 'noir', 'jet', 'onyx', 'charcoal black', 'true black',
                  'mono', 'monochrome', 'raven', 'soot', 'coal', 'pitch black', "caviar", "total eclipse"],

    # White family
    'white':     ['white', 'off white', 'ivory', 'cream', 'ecru', 'snow', 'pearl',
                  'blanc', 'off-white', 'winter white', 'bright white',
                  'cloud dancer', 'buttermilk', 'vanilla', 'eggshell', 'chalk',
                  'cotton', 'milk', 'bone white', 'porcelain', "bleach", "gardenia", "coconut shell", "fossil", 'no colour'],

    # Grey family
    'grey':      ['grey', 'gray', 'charcoal', 'silver', 'slate', 'ash', 'pewter',
                  'graphite', 'heather grey', 'light grey', 'dark grey', 'mid grey',
                  'marl grey', 'steel',
                  'ice marl', 'concrete', 'dove', 'mist', 'fog', 'smoke',
                  'platinum', 'gunmetal', 'oyster', 'cement', 'cloud', 'sliver', 'grisaille'],

    # Navy family
    'navy':      ['navy', 'naval', 'dark blue', 'midnight blue', 'dark navy',
                  'indigo', 'sapphire navy', 'deep navy', 'marine'],

    # Blue family
    'blue':      ['blue', 'cobalt', 'royal blue', 'sky blue', 'light blue', 'baby blue',
                  'denim blue', 'teal blue', 'powder blue', 'cornflower', 'azure',
                  'electric blue', 'dusty blue', 'mid blue', 'bright blue',
                  'turquoise', 'aqua', 'cerulean', 'periwinkle', 'ice blue',
                  'denim medium', 'stonewash', 'chambray', 'pacific',  "denim", "denim - medium", "petrol", 'ink', 'marina'],

    # Red family
    'red':       ['red', 'scarlet', 'crimson', 'cherry', 'ruby', 'wine red',
                  'bright red', 'dark red', 'deep red', 'fire red',
                  'tomato', 'cardinal', 'brick', 'poppy', 'flame'],

    # Pink family
    'pink':      ['pink', 'blush', 'rose', 'hot pink', 'baby pink', 'dusty pink',
                  'salmon', 'fuchsia', 'magenta', 'bubblegum', 'flamingo',
                  'bright pink', 'pale pink', 'light pink', 'dark pink', 'neon pink',
                  'raspberry', 'coral pink', 'mauve pink',
                  'petal', 'ballet', 'carnation', 'peony', 'cotton candy', "fuschia"],

    # Green family
    'green':     ['green', 'olive', 'sage', 'emerald', 'forest', 'moss', 'jade',
                  'mint', 'lime', 'teal', 'hunter green', 'dark green', 'light green',
                  'neon green', 'bright green', 'army green', 'bottle green',
                  'pistachio', 'seafoam', 'eucalyptus',
                  'chartreuse', 'birch', 'basil', 'fern', 'lawn', 'kelly green',
                  'spring green', 'viridian', 'jungle', "aloe", "laurel oak",
                  'loden frost', 'crocodile', 'laurel wreath', 'pine grove',
                  'pine', 'wasabi'],

    # Yellow family
    'yellow':    ['yellow', 'mustard', 'gold', 'lemon', 'buttercup', 'canary',
                  'sunshine', 'amber', 'marigold', 'ochre', 'saffron',
                  'bright yellow', 'pale yellow', 'neon yellow',
                  'butter', 'citron', 'honey', 'corn', 'banana', 'cream yellow', 'yarrow'],

    # Orange family
    'orange':    ['orange', 'tangerine', 'rust', 'burnt orange', 'terracotta',
                  'peach', 'apricot', 'coral', 'copper', 'ginger', 'paprika',
                  'sienna', 'cinnamon', 'pumpkin', 'amber orange',
                  'mandarin', 'persimmon', 'cantaloupe', 'tiger lily', 'sunset', 'mango', 'teracotta'],

    # Purple family
    'purple':    ['purple', 'lilac', 'lavender', 'violet', 'plum', 'mauve',
                  'grape', 'aubergine', 'amethyst', 'orchid', 'berry',
                  'burgundy purple', 'deep purple', 'bright purple',
                  'eggplant', 'iris', 'thistle', 'wisteria', 'mulberry', 'magenta purple', 'acai', 'provence'],

    # Brown family
    'brown':     ['brown', 'tan', 'chocolate', 'coffee', 'mocha', 'chestnut',
                  'caramel', 'walnut', 'toffee', 'cocoa', 'bronze', 'umber',
                  'mahogany', 'cedar', 'espresso', 'hazelnut', 'dark brown',
                  'light brown', 'mid brown',
                  'cognac', 'tobacco', 'leather', 'rust brown', 'auburn',
                  'sepia', 'biscotti', 'cinnamon brown', "cork", "toasted coconut", 'java', 'hot fudge', 'tigers eye'],

    # Beige/Neutral family
    'beige':     ['beige', 'camel', 'neutral', 'sand', 'taupe', 'oatmeal',
                  'nude', 'champagne', 'biscuit', 'fawn', 'buff', 'wheat',
                  'natural', 'stone', 'mushroom', 'mink', 'almond', 'latte',
                  'pebble', 'putty',
                  'oat', 'ecru beige', 'linen', 'flax', 'hemp', 'jute',
                  'canvas', 'doeskin', 'cashew', 'sesame', 'travertine', 'topica', 'duck egg', 'tonal', 'tonal pop', 'wood shaving', 'tofu'],

    # Khaki family
    'khaki':     ['khaki', 'olive green', 'army', 'military', 'camo', 'camouflage',
                  'dark khaki', 'light khaki',
                  'safari', 'field', 'utility', 'fatigue', 'ranger'],

    # Burgundy/Wine family
    'burgundy':  ['burgundy', 'wine', 'maroon', 'merlot', 'oxblood', 'bordeaux',
                  'claret', 'port', 'garnet', 'dark red wine',
                  'burgandy', 'cabernet', 'shiraz', 'pinot', 'cranberry'],

    # Multi/Pattern
    'multi':     ['multi', 'multicolour', 'multicolor', 'rainbow', 'tie dye',
                  'tie-dye', 'colourblock', 'color block', 'mixed', 'various',
                  'assorted', 'stripe', 'check', 'plaid', 'floral', 'print',
                  'pattern', 'abstract', 'animal', 'leopard', 'zebra', 'camo',
                  'tropical', 'paisley', 'geometric', 'gingham', 'tartan',
                  'houndstooth', 'polka dot', 'ditsy',
                  'mono spot', 'ombre', 'gradient', 'marble', 'space dye',
                  'variegated', 'heathered', 'melange', "metallic jacquard",
                  'snake', 'stinla aop', 'neon night garden', 'bright spot',
                  'alhambra', 'pastel daisy', 'tbc', 'yin & yang', 'mutli', 'dahila aop'],
}


In [ ]:
# Build reverse lookup: raw_color_token → canonical
_color_lookup = {}
for canonical, variants in CANONICAL_COLORS.items():
    for variant in variants:
        _color_lookup[variant.lower()] = canonical

def normalize_color(raw_color: str) -> str:
    """Map a raw color string to a canonical color family."""
    if pd.isna(raw_color):
        return 'other'
    color = str(raw_color).lower().strip()

    # Direct match
    if color in _color_lookup:
        return _color_lookup[color]

    # Substring match — check if any known variant is IN the color string
    # Sort by length descending to match "dark brown" before "brown"
    for variant in sorted(_color_lookup.keys(), key=len, reverse=True):
        if variant in color:
            return _color_lookup[variant]

    # Check individual words
    for word in color.split():
        if word in _color_lookup:
            return _color_lookup[word]

    return 'other'

df['color_clean'] = df['color'].str.lower().str.strip()
df['color_family'] = df['color_clean'].apply(normalize_color)

print("── COLOR NORMALIZATION ──")
print(f"Raw unique colors:  {df['color_clean'].nunique():,}")
print(f"Normalized families: {df['color_family'].nunique()}")
print(f"\n{df['color_family'].value_counts().to_string()}")
other_colors = df[df['color_family'] == 'other']['color_clean'].value_counts().head(15)
print(f"\nTop unmapped 'other' colors:\n{other_colors.to_string()}\n")

── COLOR NORMALIZATION ──
Raw unique colors:  2,929
Normalized families: 17

color_family
black       7119
multi       3443
white       3373
green       2770
pink        2301
blue        1867
beige       1557
brown       1330
grey        1287
purple      1126
orange       859
red          762
yellow       607
khaki        600
navy         482
other        300
burgundy     188

Top unmapped 'other' colors:
color_clean
no colour       3
pastel daisy    2
pine            2
wood shaving    2
fushia          2
provence        2
tbc             2
tofu            2
yin & yang      2
grisaille       2
wasabi          2
mutli           2
dahila aop      2
yarrow          2
teracotta       2



2.9  PARSE SIZE + EXTRACT AVAILABILITY

============================================================================


In [ ]:
def parse_sizes(size_str: str) -> dict:
    """Parse size string into available sizes and out-of-stock sizes."""
    if pd.isna(size_str):
        return {'available': [], 'out_of_stock': [], 'size_system': 'unknown'}

    raw = str(size_str)
    # Split by comma
    parts = [p.strip() for p in raw.split(',')]

    available = []
    oos = []
    for part in parts:
        if re.search(r'out of stock', part, re.IGNORECASE):
            # Extract just the size, remove the OOS marker
            clean = re.sub(r'\s*-?\s*out of stock\s*', '', part, flags=re.IGNORECASE).strip()
            if clean:
                oos.append(clean)
        else:
            if part.strip():
                available.append(part.strip())

    # Detect size system
    size_system = 'unknown'
    all_sizes = ' '.join(available + oos).upper()
    if 'UK' in all_sizes:
        size_system = 'UK'
    elif any(s in all_sizes for s in ['XS', 'S', 'M', 'L', 'XL', 'XXL']):
        size_system = 'letter'
    elif re.search(r'EU\s*\d', all_sizes):
        size_system = 'EU'
    elif re.search(r'US\s*\d', all_sizes):
        size_system = 'US'
    elif re.search(r'W\d+', all_sizes):
        size_system = 'waist'

    return {'available': available, 'out_of_stock': oos, 'size_system': size_system}

size_parsed = df['size'].apply(parse_sizes)
df['sizes_available']   = size_parsed.apply(lambda x: x['available'])
df['sizes_out_of_stock'] = size_parsed.apply(lambda x: x['out_of_stock'])
df['size_system']       = size_parsed.apply(lambda x: x['size_system'])
df['n_sizes_available'] = df['sizes_available'].apply(len)
df['any_in_stock']      = df['n_sizes_available'] > 0

print("── SIZE PARSING ──")
print(f"Size systems: {df['size_system'].value_counts().to_string()}")
print(f"Products fully in stock:  {(df['sizes_out_of_stock'].str.len() == 0).sum():,}")
print(f"Products with some OOS:   {(df['sizes_out_of_stock'].str.len() > 0).sum():,}")
print(f"Products fully OOS:       {(~df['any_in_stock']).sum():,}\n")

── SIZE PARSING ──
Size systems: size_system
UK         28833
letter       772
unknown      331
waist         32
EU             3
Products fully in stock:  13,259
Products with some OOS:   16,712
Products fully OOS:       0



2.10  CLEAN PRICE — HANDLE EDGE CASES

============================================================================

In [ ]:
def clean_price(price_str) -> float:
    """Extract numeric price, handling currency symbols, ranges, and sale markers."""
    if pd.isna(price_str):
        return np.nan
    raw = str(price_str).strip()
    # Remove currency symbols
    raw = re.sub(r'[£$€]', '', raw)
    # If "Now" price exists, prefer it (sale price)
    now_match = re.search(r'Now\s*(\d+\.?\d*)', raw, re.IGNORECASE)
    if now_match:
        return float(now_match.group(1))
    # If "From" price, take the starting price
    from_match = re.search(r'From\s*(\d+\.?\d*)', raw, re.IGNORECASE)
    if from_match:
        return float(from_match.group(1))
    # General numeric extraction
    match = re.search(r'(\d+\.?\d*)', raw)
    if match:
        return float(match.group(1))
    return np.nan

df['price'] = df['price'].apply(clean_price)

print("── PRICE CLEANING ──")
print(f"Null prices after cleaning: {df['price'].isna().sum()}")
print(f"Price range: £{df['price'].min():.2f} – £{df['price'].max():.2f}")
print(f"Median price: £{df['price'].median():.2f}")
print(f"Mean price: £{df['price'].mean():.2f}")

# Flag extreme outliers
q99 = df['price'].quantile(0.99)
q01 = df['price'].quantile(0.01)
outliers = ((df['price'] > q99) | (df['price'] < q01)).sum()
print(f"Price outliers (outside 1-99th percentile): {outliers}")
print()

── PRICE CLEANING ──
Null prices after cleaning: 0
Price range: £3.00 – £550.00
Median price: £32.00
Mean price: £40.74
Price outliers (outside 1-99th percentile): 541



In [ ]:
df[(df['price'] > q99) | (df['price'] < q01)]["price"]

,price
216,5.5
223,195.0
236,6.0
301,6.0
370,6.0
...,...
29385,4.0
29396,6.0
29450,225.0
29887,6.0


2.11  CLEAN SKU — CONVERT TO STRING IDENTIFIER

In [ ]:
df['sku'] = df['sku'].astype(int).astype(str)
print(f"── SKU CLEANING ──")
print(f"Unique SKUs: {df['sku'].nunique():,} (should equal row count: {len(df):,})")
print()

── SKU CLEANING ──
Unique SKUs: 29,971 (should equal row count: 29,971)



2.12  EXTRACT STYLE / VIBE TAGS FROM PRODUCT DETAILS

In [ ]:
STYLE_KEYWORDS = {
    # Occasion / Vibe
    'casual':      ['casual', 'everyday', 'relaxed', 'easy-going', 'laid-back', 'weekend',
                    'leisure', 'lounge', 'off-duty', 'comfort', 'effortless', 'low-key',
                    'chill', 'easygoing', 'comfy', 'informal', 'downtime', 'vacation',
                    'travel', 'basics', 'staples', 'go-to', 'throw-on', 'versatile'],

    'formal':      ['formal', 'occasion', 'evening', 'cocktail', 'black tie', 'event',
                    'gala', 'ceremony', 'wedding', 'party', 'special occasion', 'dressy',
                    'elegant', 'sophisticated', 'luxe', 'refined', 'polished', 'upscale',
                    'black-tie', 'white-tie', 'semi-formal', 'after-five', 'dinner',
                    'awards', 'premiere', 'red carpet', 'charity', 'fundraiser'],

    'smart_casual': ['smart casual', 'smart-casual', 'office', 'workwear', 'work-to-play',
                     'business casual', 'professional', 'corporate', 'meeting', 'workplace',
                     'boardroom', 'conference', 'presentation', 'networking', 'lunch meeting',
                     'client meeting', 'office-appropriate', 'work-ready', 'career',
                     'executive', 'professional wear', 'dress-down friday'],

    'sporty':      ['sport', 'athletic', 'active', 'performance', 'training', 'gym',
                    'running', 'workout',
                    'activewear', 'athleisure', 'sportswear', 'fitness', 'exercise',
                    'yoga', 'pilates', 'cycling', 'tennis', 'golf', 'basketball',
                    'track', 'field', 'marathon', 'outdoors', 'hiking', 'climbing',
                    'moisture-wicking', 'breathable', 'stretch', 'flexible', 'technical'],

    'boho':        ['boho', 'bohemian', 'free spirit', 'festival', 'hippie', 'folk',
                    'bohemian chic', 'wanderlust', 'nomad', 'artisan', 'eclectic',
                    'spiritual', 'earthy', 'natural', 'flowing', 'whimsical', 'dreamy',
                    'ethnic', 'tribal', 'indie', 'alternative', 'woodstock', 'gypsy',
                    'peasant', 'folklore', 'cultural', 'worldly', 'free-flowing'],

    'streetwear':  ['street', 'urban', 'skater', 'oversized', 'graphic', 'logo',
                    'hip hop', 'hypebeast', 'sneakerhead', 'supreme', 'drop', 'limited',
                    'exclusive', 'collab', 'collaboration', 'capsule', 'trendy',
                    'cool', 'fresh', 'dope', 'fire', 'sick', 'lit', 'swag', 'flex',
                    'hype', 'culture', 'youth', 'rebellious', 'contemporary'],

    'minimalist':  ['minimal', 'clean', 'understated', 'simple', 'pared-back',
                    'streamlined', 'sleek', 'modern', 'contemporary', 'timeless',
                    'architectural', 'geometric', 'monochrome', 'neutral',
                    'essentials', 'capsule wardrobe', 'investment pieces', 'quality',
                    'refined', 'sophisticated', 'less is more', 'curated', 'intentional'],

    'vintage':     ['vintage', 'retro', 'throwback', '90s', '80s', '70s', 'y2k',
                    '60s', '50s', '40s', 'nostalgic', 'classic', 'heritage', 'timeless',
                    'old school', 'antique', 'period', 'decades', 'revival', 'inspired',
                    'americana', 'mod', 'disco', 'grunge era', 'punk era', 'rockabilly',
                    'pin-up', 'victorian', 'edwardian', 'art deco', 'mid-century'],

    'glam':        ['glam', 'sparkle', 'sequin', 'glitter', 'shimmer', 'metallic', 'embellished',
                    'glamorous', 'dazzling', 'show-stopping', 'statement', 'luxurious',
                    'opulent', 'dramatic', 'eye-catching', 'rhinestone', 'crystal',
                    'beaded', 'jeweled', 'ornate', 'decorative', 'festive', 'party',
                    'celebration', 'special', 'standout', 'wow factor', 'bling'],

    'preppy':      ['preppy', 'collegiate', 'varsity', 'heritage', 'classic',
                    'ivy league', 'traditional', 'tennis club', 'country club',
                    'nautical', 'sailing', 'yacht club', 'old money', 'establishment',
                    'conservative', 'refined', 'polished', 'structured', 'blazer',
                    'button-down', 'oxford', 'khaki', 'tweed', 'argyle', 'plaid',
                    'new england', 'boarding school', 'uniform', 'academic'],

    'edgy':        ['edgy', 'punk', 'grunge', 'distressed', 'ripped', 'biker', 'gothic',
                    'rebellious', 'dark', 'moody', 'alternative', 'rock', 'metal',
                    'studded', 'spiked', 'leather', 'vinyl', 'latex', 'mesh',
                    'sheer', 'cutout', 'asymmetric', 'deconstructed', 'avant-garde',
                    'unconventional', 'bold', 'fierce', 'attitude', 'underground'],

    'romantic':    ['romantic', 'lace', 'ruffle', 'frill', 'flowy', 'dreamy', 'delicate',
                    'feminine', 'soft', 'sweet', 'pretty', 'whimsical', 'ethereal',
                    'graceful', 'elegant', 'vintage-inspired', 'victorian', 'prairie',
                    'cottagecore', 'fairytale', 'princess', 'ballerina', 'ballet',
                    'tulle', 'chiffon', 'silk', 'satin', 'pearl', 'bow', 'ribbon'],

    'coastal':     ['coastal', 'beach', 'nautical', 'maritime', 'summer', 'resort', 'holiday',
                    'vacation', 'tropical', 'island', 'seaside', 'oceanside', 'sailing',
                    'yacht', 'cruise', 'cabana', 'poolside', 'surf', 'waves',
                    'sand', 'sun', 'breeze', 'relaxed', 'carefree', 'escape',
                    'getaway', 'destination', 'paradise', 'bahamas', 'mediterranean'],

    'western':     ['western', 'cowboy', 'cowgirl', 'ranch', 'denim', 'fringe',
                    'country', 'rodeo', 'texas', 'americana', 'frontier', 'rustic',
                    'boots', 'hat', 'buckle', 'leather', 'suede', 'bandana',
                    'horseback', 'prairie', 'saloon', 'outlaw', 'sheriff',
                    'southwestern', 'desert', 'cactus', 'sunset', 'wild west'],

    # New style categories
    'cottagecore': ['cottagecore', 'cottage', 'countryside', 'pastoral', 'rural',
                    'farmhouse', 'meadow', 'picnic', 'gardening', 'homestead',
                    'rustic', 'wholesome', 'simple living', 'nature', 'organic',
                    'handmade', 'artisanal', 'traditional', 'vintage-inspired'],

    'dark_academia': ['dark academia', 'academia', 'scholarly', 'library', 'university',
                      'intellectual', 'literary', 'classical', 'gothic', 'mysterious',
                      'academic', 'bookish', 'studious', 'vintage academic', 'oxford',
                      'cambridge', 'ivy', 'tweed', 'wool', 'autumn', 'burgundy'],

    'y2k':         ['y2k', '2000s', 'millennium', 'futuristic', 'cyber', 'digital',
                    'holographic', 'iridescent', 'chrome', 'silver', 'space age',
                    'tech', 'matrix', 'rave', 'club kid', 'platform', 'low-rise'],

    'gorpcore':    ['gorpcore', 'outdoor', 'hiking', 'camping', 'mountain', 'trail',
                    'adventure', 'exploration', 'wilderness', 'rugged', 'functional',
                    'technical', 'weather-resistant', 'durable', 'practical'],

    'normcore':    ['normcore', 'ordinary', 'basic', 'unremarkable', 'dad', 'mom',
                    'tourist', 'suburban', 'middle america', 'generic', 'anti-fashion',
                    'comfort', 'practicality', 'simplicity', 'everyday'],

    # Fit descriptors
    'oversized':   ['oversized', 'boxy', 'relaxed fit', 'loose',
                    'baggy', 'slouchy', 'roomy', 'generous', 'flowing', 'voluminous',
                    'dropped shoulder', 'boyfriend', 'dad', 'chunky', 'cozy',
                    'comfortable', 'laid-back', 'effortless', 'casual fit'],

    'fitted':      ['fitted', 'slim', 'skinny', 'tailored', 'figure-hugging', 'body-con',
                    'tight', 'snug', 'form-fitting', 'sculpting', 'second skin',
                    'stretch', 'bandage', 'curve-hugging', 'streamlined', 'sleek',
                    'structured', 'shaped', 'contoured', 'molded', 'precision fit'],

    'cropped':     ['cropped', 'crop',
                    'short', 'abbreviated', 'mini', 'truncated', 'cut short',
                    'belly-baring', 'midriff', 'high-low', 'asymmetric hem',
                    'above waist', 'waist-length', 'boxy crop', 'fitted crop'],

    'flowy':       ['flowy', 'flowing', 'fluid', 'draping', 'cascading', 'billowing',
                    'swishy', 'movement', 'graceful', 'airy', 'light', 'ethereal',
                    'floating', 'dance-like', 'breeze', 'soft drape'],

    # Pattern/Detail tags
    'floral':      ['floral', 'flower', 'ditsy', 'botanical',
                    'bloom', 'blossom', 'rose', 'daisy', 'lily', 'peony', 'tulip',
                    'garden', 'meadow', 'spring', 'romantic', 'feminine',
                    'micro floral', 'large floral', 'abstract floral', 'vintage floral',
                    'tropical floral', 'watercolor floral', 'embroidered floral'],

    'animal_print': ['animal', 'leopard', 'snake', 'zebra', 'tiger',
                     'cheetah', 'jaguar', 'python', 'crocodile', 'alligator',
                     'cow', 'dalmatian', 'giraffe', 'wild', 'safari', 'jungle',
                     'predator', 'fierce', 'bold', 'statement', 'exotic'],

    'striped':     ['stripe', 'striped', 'pinstripe', 'breton',
                    'horizontal stripe', 'vertical stripe', 'diagonal stripe',
                    'wide stripe', 'thin stripe', 'bold stripe', 'subtle stripe',
                    'multi-stripe', 'rainbow stripe', 'nautical stripe', 'classic stripe'],

    'plaid':       ['plaid', 'check', 'tartan', 'gingham', 'houndstooth',
                    'buffalo check', 'windowpane', 'madras', 'glen plaid',
                    'scottish', 'clan', 'traditional', 'heritage', 'preppy',
                    'school uniform', 'country', 'rustic', 'classic pattern'],

    'cutout':      ['cutout', 'cut-out', 'peek-a-boo', 'open back',
                    'keyhole', 'cold shoulder', 'shoulder cutout', 'side cutout',
                    'midriff cutout', 'strategic cutout', 'geometric cutout',
                    'laser cut', 'perforated', 'eyelet', 'lattice', 'caged'],

    'geometric':   ['geometric', 'abstract', 'modern', 'architectural', 'linear',
                    'angular', 'triangular', 'circular', 'hexagonal', 'diamond',
                    'chevron', 'zigzag', 'optical', 'graphic', 'bold pattern'],

    'tie_dye':     ['tie dye', 'tie-dye', 'dyed', 'ombre', 'gradient', 'rainbow',
                    'psychedelic', 'hippie', '60s', 'festival', 'bohemian',
                    'swirl', 'spiral', 'sunburst', 'bullseye', 'marble'],

    'polka_dot':   ['polka dot', 'dot', 'spotted', 'dalmatian', 'pin dot',
                    'micro dot', 'large dot', 'retro', 'vintage', 'playful',
                    'whimsical', '50s', 'rockabilly', 'pin-up', 'classic'],

    # Texture/Fabric details
    'textured':    ['textured', 'texture', 'tactile', 'dimensional', 'raised',
                    'embossed', 'quilted', 'ribbed', 'cable knit', 'honeycomb',
                    'waffle', 'bouclé', 'tweed', 'corduroy', 'velvet', 'crushed'],

    'sheer':       ['sheer', 'transparent', 'see-through', 'mesh', 'net',
                    'tulle', 'chiffon', 'organza', 'voile', 'gauze', 'lace',
                    'delicate', 'layering', 'overlay', 'ethereal', 'romantic'],

    'metallic':    ['metallic', 'metal', 'foil', 'lamé', 'holographic', 'iridescent',
                    'chrome', 'silver', 'gold', 'copper', 'bronze', 'pewter',
                    'reflective', 'mirror', 'liquid metal', 'space age', 'futuristic']
}

In [ ]:
def extract_style_tags(name: str, details: str) -> list:
    """Extract style/vibe tags from product name and description."""
    text = f"{str(name)} {str(details)}".lower()
    tags = []
    for tag, keywords in STYLE_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            tags.append(tag)
    return tags

df['style_tags'] = df.apply(
    lambda r: extract_style_tags(r['name'], r.get('product_details', '')), axis=1
)
df['n_style_tags'] = df['style_tags'].apply(len)

print("── STYLE TAG EXTRACTION ──")
all_tags = [tag for tags in df['style_tags'] for tag in tags]
tag_counts = Counter(all_tags)
print(f"Total unique style tags: {len(tag_counts)}")
print(f"Top 15 tags:\n{pd.Series(tag_counts).sort_values(ascending=False).head(15).to_string()}")
print(f"Products with ≥1 tag: {(df['n_style_tags'] > 0).sum():,} / {len(df):,}")
print()

── STYLE TAG EXTRACTION ──
Total unique style tags: 34
Top 15 tags:
streetwear    11063
cropped        9929
romantic       7246
fitted         5507
oversized      5068
casual         4823
coastal        4368
sheer          3380
floral         3187
edgy           3029
cutout         2912
textured       2603
western        2417
preppy         2147
glam           1973
Products with ≥1 tag: 28,554 / 29,971



2.13  BUILD COMPOSITE SEARCH TEXT FOR NLP / EMBEDDINGS

 ============================================================================

In [ ]:
def build_search_text(row) -> str:
    """
    Build a rich composite text optimized for semantic search.
    This combines all relevant text fields into a single searchable string.
    CLIP text encoder will embed this for text-to-text matching.
    Also useful for traditional NLP (TF-IDF, BM25) fallback.
    """
    parts = []

    # Product name (most important — weight it by repetition)
    if pd.notna(row.get('name')):
        parts.append(str(row['name']))

    # Category
    cat = row.get('category_extracted', '')
    if cat and cat != 'Other':
        parts.append(cat)

    # Color
    color = row.get('color_clean', '')
    if color:
        parts.append(color)

    # Color family (for broader matching)
    color_fam = row.get('color_family', '')
    if color_fam and color_fam != 'other':
        parts.append(color_fam)

    # Style tags as natural language
    tags = row.get('style_tags', [])
    if tags:
        parts.append(' '.join(tags))

    # Product details (cleaned description)
    details = row.get('product_details', '')
    if details:
        parts.append(details)

    # Fabric/materials
    materials = row.get('materials', [])
    if materials:
        parts.append(' '.join(materials))

    # Gender
    gender = row.get('gender', '')
    if gender and gender != 'Unisex':
        parts.append(f"{gender}'s fashion")

    # Brand
    brand = row.get('brand', '')
    if brand and brand != 'Unknown':
        parts.append(brand)

    return ' | '.join(filter(None, parts))

df['search_text'] = df.apply(build_search_text, axis=1)
print("── COMPOSITE SEARCH TEXT ──")
print(f"Mean search text length: {df['search_text'].str.len().mean():.0f} chars")
print(f"Sample:\n{df['search_text'].iloc[0][:200]}...\n")

── COMPOSITE SEARCH TEXT ──
Mean search text length: 263 chars
Sample:
Miss Selfridge Petite rib knit frill hem funnel dress with heart buttons in black | Dresses | black | black | romantic fitted | Petite by Miss Selfridge Petite The scroll is over High neck Long sleeve...



2.14  PRICE TIER BINNING (for search filters)

============================================================================

In [ ]:
df['price_tier'] = pd.cut(
    df['price'],
    bins=[0, 15, 30, 50, 80, 120, 200, float('inf')],
    labels=['Under £15', '£15-30', '£30-50', '£50-80', '£80-120', '£120-200', '£200+'],
    include_lowest=True
)
print("── PRICE TIERS ──")
print(df['price_tier'].value_counts().sort_index().to_string())
print()

── PRICE TIERS ──
price_tier
Under £15     3698
£15-30       10780
£30-50        8712
£50-80        4233
£80-120       1663
£120-200       729
£200+          156



2.15  FINAL COLUMN SELECTION AND EXPORT

============================================================================

In [ ]:
# Select the clean, structured columns for the search engine
FINAL_COLUMNS = [
    # Identifiers
    'sku', 'name', 'brand', 'url',
    # Core attributes
    'price', 'price_tier',
    'color_clean', 'color_family',
    'category_extracted', 'gender',
    # Parsed description fields
    'product_details', 'fabric_raw', 'materials',
    'fit_info', 'care_info',
    # Style & search
    'style_tags', 'search_text',
    # Size & availability
    'sizes_available', 'sizes_out_of_stock', 'size_system',
    'any_in_stock', 'n_sizes_available',
    # Images
    'image_urls', 'n_images', 'primary_image_url',
    # Flags
    '_url_mismatch',
]

df_clean = df[FINAL_COLUMNS].copy()
df_clean = df_clean.rename(columns={
    'category_extracted': 'category',
    '_url_mismatch': 'url_mismatch_flag',
})

print("="*78)
print(f"FINAL CLEAN DATASET: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print("="*78)
print(f"\nColumns: {list(df_clean.columns)}")
print(f"\nData types:\n{df_clean.dtypes.to_string()}")

# Save
df_clean.to_parquet('asos_clean.parquet', index=False)
df_clean.to_csv('asos_clean.csv', index=False)
print(f"\n✅ Saved: asos_clean.parquet and asos_clean.csv")

FINAL CLEAN DATASET: 29,971 rows × 26 columns

Columns: ['sku', 'name', 'brand', 'url', 'price', 'price_tier', 'color_clean', 'color_family', 'category', 'gender', 'product_details', 'fabric_raw', 'materials', 'fit_info', 'care_info', 'style_tags', 'search_text', 'sizes_available', 'sizes_out_of_stock', 'size_system', 'any_in_stock', 'n_sizes_available', 'image_urls', 'n_images', 'primary_image_url', 'url_mismatch_flag']

Data types:
sku                     object
name                    object
brand                   object
url                     object
price                  float64
price_tier            category
color_clean             object
color_family            object
category                object
gender                  object
product_details         object
fabric_raw              object
materials               object
fit_info                object
care_info               object
style_tags              object
search_text             object
sizes_available         object
size

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# df_clean.to_parquet('/content/drive/MyDrive/Colab Notebooks/asos_clean.parquet', index=False)
# df_clean.to_csv('/content/drive/MyDrive/Colab Notebooks/asos_clean.csv', index=False)